<div align="center">
    <h1>Deterministic Pharmaceutical Manufacturing Optimization</h1>
    <a href="https://github.com/sa1K">Sai Karthik</a>
    <br>
    <i>Weldon School of Biomedical Engineering, Purdue University</i>
    <br>
    <br>
    <a href="https://github.com/parkyr">Yirang Park</a>
    <br>
    <i>Davidson School of Chemical Engineering, Purdue University</i>
    <br>
    <br>
    <a href="https://github.com/bernalde">David E. Bernal Neira</a>
    <br>
    <i>Davidson School of Chemical Engineering, Purdue University</i>
    <br>
    <br>
    <a href="https://secquoia.github.io/">
        <img src="https://img.shields.io/badge/🌲⚛️🌐-SECQUOIA-blue" alt="SECQUOIA"/>
    </a>
</div>

# Introduction

This notebook demonstrates the deterministic optimization model for pharmaceutical manufacturing networks. The model selects optimal vendors at each manufacturing step and determines transportation routes to minimize total cost while meeting demand requirements.

# Setup and Imports

In [ ]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

# Install dependencies if in Colab
if IN_COLAB:
    !pip install -q pyomo
    !apt-get install -y -qq glpk-utils
    !pip install highspy
    !sudo apt-get install graphviz graphviz-dev
    !pip install networkx

In [ ]:
import pyomo.environ as pyo
import matplotlib.pyplot as plt
import pandas as pd
import time

# Import from our package
from pharma_optimizer import (
    Generator,
    EnhancedProductionOptimizer,
    visualize_solution_from_optimizer
)

# Data Generation

Generate random manufacturing and transportation data for the optimization problem.

In [ ]:
# Generate a 3-step, 3-vendor instance
generator = Generator(steps=3, options=3)
generator.createManufacturingData()
generator.createTransportData()

In [ ]:
# Load and display production data
prod_df = pd.read_csv("manufacturingData2.csv")
print("Production Data:")
print(prod_df)
print("\n" + "="*50)

# Load and display transport data
transport_df = pd.read_csv("transportData2.csv")
print("\nTransport Data:")
print(transport_df)

# Run Optimization

Solve the deterministic optimization problem to find the minimum cost vendor selection and transportation routing.

In [ ]:
# Create and solve the optimization model
optimizer = EnhancedProductionOptimizer(
    "manufacturingData2.csv",
    "transportData2.csv",
    demand=100
)

# Solve using GLPK solver
if IN_COLAB:
    opt = pyo.SolverFactory('glpk', executable='/usr/bin/glpsol')
else:
    opt = pyo.SolverFactory('glpk')

result = opt.solve(optimizer.model, tee=False)
print(f"Solver status: {result.solver.termination_condition}")

# Display results
optimizer.summary()

# Visualization

Visualize the optimal solution path through the manufacturing network.

In [ ]:
# Generate and display the visualization
G, pos, chosen_path = visualize_solution_from_optimizer(optimizer)
plt.show()

print(f"\nVisualization complete!")
print(f"Found {len(optimizer.vendors)} steps with {sum(len(v) for v in optimizer.vendors.values())} total vendor options")
print(f"Selected path has {len(chosen_path)} steps")

# Solver Scaling Analysis

Compare solver performance across different problem sizes.

In [ ]:
# Scaling: vary both steps and vendors
time_to_run_glpk = []
time_to_run_highs = []

for i in range(5, 35, 5):
    generator = Generator(i, i)
    generator.createManufacturingData()
    generator.createTransportData()

    optimizer = EnhancedProductionOptimizer(
        "manufacturingData2.csv",
        "transportData2.csv",
        demand=100
    )

    # GLPK
    opt = pyo.SolverFactory('glpk')
    result = opt.solve(optimizer.model, tee=False)
    time_to_run_glpk.append(result.solver.time)

    # HiGHS
    opt = pyo.SolverFactory('highs')
    t0 = time.perf_counter()
    result = opt.solve(optimizer.model, tee=False)
    time_to_run_highs.append(time.perf_counter() - t0)

    print(f"Size {i}: GLPK={time_to_run_glpk[-1]:.3f}s, HiGHS={time_to_run_highs[-1]:.3f}s")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(range(5, 35, 5), time_to_run_glpk, label="GLPK")
ax.scatter(range(5, 35, 5), time_to_run_highs, label="HiGHS")
ax.set_xlabel("Number of vendors and steps", fontsize=14)
ax.set_ylabel("Solution time (seconds)", fontsize=14)
ax.set_title("Solution time for varying both number of vendors and steps", fontsize=14)
ax.legend(fontsize=12)
plt.show()

In [ ]:
# Scaling: vary only steps (fixed 5 vendors)
time_to_run_glpk = []
time_to_run_highs = []

for i in range(5, 35, 5):
    generator = Generator(i, 5)
    generator.createManufacturingData()
    generator.createTransportData()

    optimizer = EnhancedProductionOptimizer(
        "manufacturingData2.csv",
        "transportData2.csv",
        demand=100
    )

    opt = pyo.SolverFactory('glpk')
    result = opt.solve(optimizer.model, tee=False)
    time_to_run_glpk.append(result.solver.time)

    opt = pyo.SolverFactory('highs')
    t0 = time.perf_counter()
    result = opt.solve(optimizer.model, tee=False)
    time_to_run_highs.append(time.perf_counter() - t0)

    print(f"Steps {i}: GLPK={time_to_run_glpk[-1]:.3f}s, HiGHS={time_to_run_highs[-1]:.3f}s")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(range(5, 35, 5), time_to_run_glpk, label="GLPK")
ax.scatter(range(5, 35, 5), time_to_run_highs, label="HiGHS")
ax.set_xlabel("Number of steps", fontsize=14)
ax.set_ylabel("Solution time (seconds)", fontsize=14)
ax.set_title("Solution time for varying number of steps (5 vendors)", fontsize=14)
ax.legend(fontsize=12)
plt.show()

In [ ]:
# Scaling: vary only vendors (fixed 5 steps)
time_to_run_glpk = []
time_to_run_highs = []

for i in range(5, 35, 5):
    generator = Generator(5, i)
    generator.createManufacturingData()
    generator.createTransportData()

    optimizer = EnhancedProductionOptimizer(
        "manufacturingData2.csv",
        "transportData2.csv",
        demand=100
    )

    opt = pyo.SolverFactory('glpk')
    result = opt.solve(optimizer.model, tee=False)
    time_to_run_glpk.append(result.solver.time)

    opt = pyo.SolverFactory('highs')
    t0 = time.perf_counter()
    result = opt.solve(optimizer.model, tee=False)
    time_to_run_highs.append(time.perf_counter() - t0)

    print(f"Vendors {i}: GLPK={time_to_run_glpk[-1]:.3f}s, HiGHS={time_to_run_highs[-1]:.3f}s")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(range(5, 35, 5), time_to_run_glpk, label="GLPK")
ax.scatter(range(5, 35, 5), time_to_run_highs, label="HiGHS")
ax.set_xlabel("Number of vendors per step", fontsize=14)
ax.set_ylabel("Solution time (seconds)", fontsize=14)
ax.set_title("Solution time for varying number of vendors (5 steps)", fontsize=14)
ax.legend(fontsize=12)
plt.show()